In [1]:
!pip install -q openai-whisper gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.0 MB/s eta 0:00:00


In [2]:
import whisper
import gradio as gr

print("Whisper and Gradio imported successfully!")

Whisper and Gradio imported successfully!


In [3]:
from google.colab import files

uploaded = files.upload()

Saving freesound_community-shortfilm-voice-56795.mp3 to freesound_community-shortfilm-voice-56795.mp3


In [4]:
import whisper

model = whisper.load_model("base")

print("Whisper model loaded successfully!")

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 280MiB/s]


Whisper model loaded successfully!


In [5]:
import os

audio_file = list(uploaded.keys())[0]

print("Audio file:", audio_file)

Audio file: freesound_community-shortfilm-voice-56795.mp3


In [6]:
result = model.transcribe(audio_file)

transcript = result["text"]

print("===== MEETING TRANSCRIPT =====")
print(transcript)

/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


===== MEETING TRANSCRIPT =====
 They run, they laugh. I see the glow shining in their eyes, not like hers. She seems distant, strange, somehow cold. A couple of days after I receive the call. I curse, scream and cry. They're gone. A drink and cry and dream over and over. Why? Time breaks me, expanding days, months or maybe years. But the pain still remains. It grows. It changes me. Someone told me she got released from the Psychiatric ward. 426 days after. My head spins. I got confused. The loneliness. It's time. The road becomes endless. I feel the cold wind on my face, my eyes burn. Got to the house. It all looks the same. What can he hear them laughing like they were no stolt staken? And then she comes. She sees me with kindness in her eyes. She looks at the flowers and she says she still loves me. Those words felt me like a razor blade. Goodbye, my love.


In [7]:
clean_transcript = transcript.strip()

print("===== CLEAN MEETING TRANSCRIPT =====")
print(clean_transcript)

===== CLEAN MEETING TRANSCRIPT =====
They run, they laugh. I see the glow shining in their eyes, not like hers. She seems distant, strange, somehow cold. A couple of days after I receive the call. I curse, scream and cry. They're gone. A drink and cry and dream over and over. Why? Time breaks me, expanding days, months or maybe years. But the pain still remains. It grows. It changes me. Someone told me she got released from the Psychiatric ward. 426 days after. My head spins. I got confused. The loneliness. It's time. The road becomes endless. I feel the cold wind on my face, my eyes burn. Got to the house. It all looks the same. What can he hear them laughing like they were no stolt staken? And then she comes. She sees me with kindness in her eyes. She looks at the flowers and she says she still loves me. Those words felt me like a razor blade. Goodbye, my love.


In [8]:
!pip install -q openai

In [9]:
!pip install -q transformers accelerate

In [10]:
from transformers import pipeline

print("Loading AI model...")

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

print("AI model loaded successfully!")

Loading AI model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

AI model loaded successfully!


In [11]:
prompt = f"""
Analyze the following transcript.

Return the result in these sections:

1. Summary
2. Key Points
3. Decisions
4. Action Items
5. Deadlines

Transcript:
{clean_transcript}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

output = generator(
    messages,
    max_new_tokens=500,
    do_sample=False
)

print(output[0]["generated_text"][-1]["content"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


### Summary
The summary is provided below:

- **Summary:** The speaker describes feeling lost and alone after receiving news that someone close has left. They recall a period of intense emotional distress, including crying and dreaming about the person who left. The speaker mentions being confused and lonely, but also feels relief when they learn the other person was released from a psychiatric ward. The final lines express sadness and longing for the departed person.

### Key Points
1. **Emotional Distress:**
   - The speaker experienced intense emotional turmoil, including crying, screaming, and dreaming.
   
2. **Recovery Process:**
   - After some time, the speaker began to feel better, though the pain remained.
   
3. **Psychological Impact:**
   - The experience caused confusion and loneliness, which eventually led to a sense of relief upon learning the other person had been released from a psychiatric ward.
   
4. **Lingering Emotions:**
   - Despite the initial shock and confus

In [12]:
import gradio as gr

def meetmind(audio_file):
    # Transcribe audio
    result = model.transcribe(audio_file)
    transcript = result["text"]

    # Create AI prompt
    prompt = f"""
Analyze this meeting transcript.

Give:
1. Summary
2. Key Points
3. Decisions
4. Action Items
5. Deadlines

If something is not mentioned, write "None mentioned."

Transcript:
{transcript}
"""

    # Generate AI analysis
    output = generator(
        prompt,
        max_new_tokens=400,
        do_sample=False
    )

    analysis = output[0]["generated_text"]

    return transcript, analysis

In [13]:
demo = gr.Interface(
    fn=meetmind,
    inputs=gr.Audio(
        type="filepath",
        label="Upload Meeting Audio"
    ),
    outputs=[
        gr.Textbox(
            label="Meeting Transcript",
            lines=15
        ),
        gr.Textbox(
            label="AI Meeting Analysis",
            lines=20
        )
    ],
    title="MeetMind - AI Meeting Intelligence",
    description="Upload a meeting recording to generate a transcript, summary, key points, decisions, action items, and deadlines."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d418e126bf690c7ea8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
